In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance

In [7]:
df = pd.read_csv("../data/ACTUAL_qol.csv")
df['age_health_interaction'] = df['age'] * df['health_new']
df['years_married'] = (df['year'] - df['agewed']).fillna(0) # years since marriage (Year - age when wed)
df['is_very_happy'] = (df['happy_new'] == 2).astype(int)
df.head(5)

,year,id,age,sex_new,race_new,degree_new,educ,relig,marital,agewed,divorce,widowed,wrkstat,happy_new,hapmar,health_new,life_new,age_health_interaction,years_married,is_very_happy
0,1973,1,54,0,0,0,6,1,1,29.0,2.0,NaN,1,0,1.0,2,2,108,1944.0,0
1,1973,2,51,1,0,0,8,1,1,21.0,2.0,NaN,7,2,1.0,3,2,153,1952.0,1
2,1973,3,36,1,0,0,11,1,1,17.0,2.0,NaN,1,1,1.0,4,2,144,1956.0,0
3,1973,4,32,0,0,1,12,1,1,27.0,2.0,NaN,1,1,1.0,4,2,128,1946.0,0
4,1973,5,54,1,0,0,8,1,1,18.0,2.0,NaN,7,1,2.0,3,2,162,1955.0,0


Baseline model

In [16]:
# Define features and target
X1 = df.drop(columns=['happy_new', 'is_very_happy', 'id']).fillna(-1)
y1 = df['happy_new']

# Split and train
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, random_state=42)
rf = RandomForestClassifier(
    n_estimators=300, 
    class_weight='balanced',  # Fixes the class imbalance problem
    max_depth=15,              # Prevents the model from "over-memorizing" noise
    min_samples_leaf=5,        # Ensures the model generalizes better
    random_state=42, 
    n_jobs=-1
)
rf.fit(X_train1, y_train1)

# Predict
preds1 = rf.predict(X_test1)
preds1

print(accuracy_score(y_test1, preds1))
print(classification_report(y_test1, preds1))
print(confusion_matrix(y_test1, preds1))

# Cross-Validation --> model works across different subsets of data
cv_scores = cross_val_score(rf, X_train1, y_train1, cv=10)
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")

# Permutation Importance --> "Explainability"
result = permutation_importance(rf, X_test1, y_test1, n_repeats=10, random_state=42)

# results
importance_df = pd.DataFrame({
    'Feature': X1.columns,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

print(importance_df.head(10))

0.5968281501672655
              precision    recall  f1-score   support

           0       0.39      0.48      0.43      1133
           1       0.71      0.57      0.63      4457
           2       0.56      0.70      0.62      2481

    accuracy                           0.60      8071
   macro avg       0.55      0.58      0.56      8071
weighted avg       0.62      0.60      0.60      8071

[[ 542  448  143]
 [ 703 2531 1223]
 [ 134  603 1744]]
Mean CV Accuracy: 0.6015
                   Feature  Importance
12                  hapmar    0.075307
14                life_new    0.036129
7                  marital    0.010048
11                 wrkstat    0.004721
13              health_new    0.004113
2                  sex_new    0.002726
6                    relig    0.000050
15  age_health_interaction    0.000037
3                 race_new   -0.000062
5                     educ   -0.000248


New forest

In [12]:
# Define features and target
X = df.drop(columns=['happy_new', 'is_very_happy', 'id', 'divorce', 'widowed']).fillna(-1)
y = df['is_very_happy']

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_improved = RandomForestClassifier(
    n_estimators=300, 
    class_weight='balanced',  # Fixes the class imbalance problem
    max_depth=15,              # Prevents the model from "over-memorizing" noise
    min_samples_leaf=5,        # Ensures the model generalizes better
    random_state=42, 
    n_jobs=-1
)
rf_improved.fit(X_train, y_train)

# Predict
predictions = rf_improved.predict(X_test)
predictions

array([1, 0, 1, ..., 1, 0, 1], shape=(8071,))

In [17]:
# accuracy score
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

# confusion matrix
print(confusion_matrix(y_test, predictions))

0.7369594845744022
              precision    recall  f1-score   support

           0       0.85      0.75      0.80      5590
           1       0.56      0.70      0.62      2481

    accuracy                           0.74      8071
   macro avg       0.70      0.73      0.71      8071
weighted avg       0.76      0.74      0.74      8071

[[4201 1389]
 [ 734 1747]]


In [18]:
# Cross-Validation --> model works across different subsets of data
cv_scores = cross_val_score(rf_improved, X_train, y_train, cv=10)
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")

# Permutation Importance --> "Explainability"
result = permutation_importance(rf_improved, X_test, y_test, n_repeats=10, random_state=42)

# results
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

print(importance_df.head(10))

Mean CV Accuracy: 0.7418
                   Feature  Importance
10                  hapmar    0.073882
7                  marital    0.034469
12                life_new    0.021298
11              health_new    0.005142
13  age_health_interaction    0.001859
6                    relig    0.001574
2                  sex_new    0.001351
9                  wrkstat    0.000149
3                 race_new   -0.000260
0                     year   -0.001127
